In [18]:
import pandas as pd
import geopandas as gpd

In [19]:
survey_path = "../../results/surveys/edgt_lyon"
spatial_path = "../../results/surveys/edgt_lyon/trips.geoparquet"
output_path = "../../results/road/survey.parquet"

In [20]:
# Load survey data
df_persons = pd.read_parquet("{}/persons.parquet".format(survey_path))
df_trips = pd.read_parquet("{}/trips.parquet".format(survey_path))
df_legs = pd.read_parquet("{}/legs.parquet".format(survey_path))
df_spatial = gpd.read_parquet(spatial_path)

In [21]:
# Prepare spatial data
df_spatial["origin_x"] = gpd.GeoSeries.from_wkt(df_spatial["origin_geometry"]).x
df_spatial["origin_y"] = gpd.GeoSeries.from_wkt(df_spatial["origin_geometry"]).y
df_spatial["destination_x"] = gpd.GeoSeries.from_wkt(df_spatial["destination_geometry"]).x
df_spatial["destination_y"] = gpd.GeoSeries.from_wkt(df_spatial["destination_geometry"]).y

In [22]:
df_trips.loc[df_trips["mode"] == "car"]

,household_id,person_id,trip_id,mode,mode_code,euclidean_distance,travel_time,departure_time,origin_cell,destination_cell,origin_activity_type,destination_activity_type,is_valid
6,1,4,6,car,21,10470,1200.0,23400.0,101001,252004,home,work,True
7,1,4,7,car,21,10470,3600.0,63000.0,252004,101001,work,home,True
10,2,6,10,car,21,32850,4500.0,50400.0,101002,704009,home,home,True
11,2,6,11,car,21,32850,4500.0,64800.0,704009,101002,home,home,True
57,12,21,57,car,21,5820,1500.0,27900.0,101002,217001,home,work,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99580,16359,36561,99580,car,21,3610,600.0,61200.0,712451,711006,home,leisure,True
99581,16359,36561,99581,car,21,3610,600.0,61800.0,711006,712451,leisure,home,True
99582,16360,36562,99582,car,21,770,300.0,27000.0,712451,712001,home,other,True
99583,16360,36562,99583,car,21,770,480.0,27420.0,712001,712451,other,work,True


In [23]:
# Only keep valid car trips
df_trips = df_trips[
    (df_trips["mode"] == "car") & (df_trips["mode_code"] != 81) & df_trips["is_valid"]
].copy()

# Merge in weight
df_trips = pd.merge(df_trips, df_persons[["person_id", "weight"]])

# Merge in spatial data
df_trips = pd.merge(df_trips, df_spatial)[[
    "trip_id",
    "origin_x", "origin_y",
    "destination_x", "destination_y",
    "travel_time", "departure_time", "weight",
    "euclidean_distance", "computed_distance", "distance_error"
]].rename(columns = {
    "travel_time": "survey_travel_time_s"
})

In [24]:
df_trips["distance_error"].describe()

count    39887.000000
mean       584.248944
std        632.919601
min          0.083983
25%        167.674404
50%        390.001633
75%        779.561776
max      18898.408545
Name: distance_error, dtype: float64

In [25]:
# Output
df_trips.to_parquet(output_path)